In [1]:
import numpy as np
from scipy.spatial.transform import Rotation as R
from scipy.optimize import least_squares

class EllipsoidParams:
    def __init__(self, center, radii, rotation_matrix):
        self.center = center
        self.radii = radii
        self.rotation_matrix = rotation_matrix

    def to_vector(self):
        # Serialize: Center (3) + Radii (3) + Rotation (4, Quaternion)
        q = R.from_matrix(self.rotation_matrix).as_quat()
        return np.concatenate([self.center, self.radii, q])

    @staticmethod
    def from_vector(vec):
        center = vec[0:3]
        radii = vec[3:6]
        # Normalize quaternion to ensure valid rotation
        quat = vec[6:10] / np.linalg.norm(vec[6:10])
        rot = R.from_quat(quat).as_matrix()
        return EllipsoidParams(center, radii, rot)

class CompressionNode:
    def __init__(self, depth):
        self.depth = depth
        self.ellipsoid = None
        self.left = None
        self.right = None
        self.is_leaf = False
        # Leaf data
        self.residuals_r = None
        self.residuals_theta = None
        self.residuals_phi = None

def fit_ellipsoid(points):
    """
    Step 1: PCA for initial guess.
    Step 2: Non-linear optimization to snap ellipsoid to surface.
    """
    if len(points) < 4:
        # Fallback for too few points: unit sphere at centroid
        return EllipsoidParams(np.mean(points, axis=0), np.ones(3), np.eye(3))

    # --- 1. PCA Initialization ---
    center = np.mean(points, axis=0)
    centered_points = points - center
    cov = np.cov(centered_points, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)

    # Sort eigenvectors by eigenvalue (descending)
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    # Initial radii (sqrt of variance covers ~1 sigma)
    # We multiply by 2.0 to cover more points initially
    radii = 2.0 * np.sqrt(np.maximum(eigvals, 1e-6))
    rotation = eigvecs

    # --- 2. Optimization (Surface Fitting) ---
    # We optimize 9 parameters: Center(3), LogRadii(3), RotationVector(3)
    # Using LogRadii ensures radii stay positive.

    def transform_to_unit(p, c, r_log, rot_vec):
        # p: (N, 3)
        # rot_vec -> matrix
        rot_mat = R.from_rotvec(rot_vec).as_matrix()
        radii_val = np.exp(r_log)

        # World -> Local: R.T * (P - C) / S
        p_centered = p - c
        p_rotated = p_centered @ rot_mat # essentially dot(p, R) if R is column-vecs
        p_scaled = p_rotated / radii_val
        return p_scaled

    def loss_function(params, p_target):
        # Params: cx, cy, cz, log_rx, log_ry, log_rz, rx, ry, rz
        c = params[0:3]
        r_log = params[3:6]
        rot_vec = params[6:9]

        p_unit = transform_to_unit(p_target, c, r_log, rot_vec)
        # We want magnitude to be close to 1.0 (Surface fitting)
        distances = np.linalg.norm(p_unit, axis=1)
        return distances - 1.0

    # Initial param vector
    r_vec_init = R.from_matrix(rotation).as_rotvec()
    x0 = np.concatenate([center, np.log(radii), r_vec_init])

    # Run Levenberg-Marquardt
    res = least_squares(loss_function, x0, args=(points,), method='lm', max_nfev=20)

    # Extract optimized parameters
    opt_c = res.x[0:3]
    opt_radii = np.exp(res.x[3:6])
    opt_rot = R.from_rotvec(res.x[6:9]).as_matrix()

    return EllipsoidParams(opt_c, opt_radii, opt_rot)

def world_to_local(points, ellipsoid):
    """
    Transform World Points -> Local Unit Sphere Space
    """
    # Translate
    p = points - ellipsoid.center
    # Rotate (Inverse rotation is Transpose for rotation matrices)
    p = p @ ellipsoid.rotation_matrix
    # Scale
    p = p / ellipsoid.radii
    return p

def local_to_world(local_points, ellipsoid):
    """
    Transform Local Unit Sphere Points -> World Space
    """
    # Scale
    p = local_points * ellipsoid.radii
    # Rotate back
    p = p @ ellipsoid.rotation_matrix.T
    # Translate back
    p = p + ellipsoid.center
    return p

def cartesian_to_spherical(xyz):
    """ Returns r, theta (polar), phi (azimuth) """
    r = np.linalg.norm(xyz, axis=1)
    # Clip z/r to [-1, 1] to avoid nan in arccos
    theta = np.arccos(np.clip(xyz[:, 2] / (r + 1e-9), -1.0, 1.0))
    phi = np.arctan2(xyz[:, 1], xyz[:, 0])
    return r, theta, phi

def spherical_to_cartesian(r, theta, phi):
    x = r * np.sin(theta) * np.cos(phi)
    y = r * np.sin(theta) * np.sin(phi)
    z = r * np.cos(theta)
    return np.stack([x, y, z], axis=1)

# --- Main Recursive Functions ---

def build_tree(points, current_depth, max_depth):
    node = CompressionNode(current_depth)

    # 1. Fit Ellipsoid
    node.ellipsoid = fit_ellipsoid(points)

    # 2. Transform points to Local Space (Residuals)
    local_points = world_to_local(points, node.ellipsoid)

    # Check termination
    if current_depth >= max_depth or len(points) <= 2:
        node.is_leaf = True
        # Convert final residuals to spherical
        r, theta, phi = cartesian_to_spherical(local_points)
        node.residuals_r = r
        node.residuals_theta = theta
        node.residuals_phi = phi
        return node

    # 3. Split
    # Simple Split Strategy: Sort along the longest axis (X in local space usually corresponds to largest variance if PCA worked well)
    # We sort by the X coordinate of the *local* points.
    split_axis = 0
    # sort_indices = np.argsort(local_points[:, split_axis])
    # sorted_points = points[sort_indices] # Sort original points to keep consistency
    sorted_points = points

    mid = len(sorted_points) // 2
    left_points = sorted_points[:mid]
    right_points = sorted_points[mid:]

    node.left = build_tree(left_points, current_depth + 1, max_depth)
    node.right = build_tree(right_points, current_depth + 1, max_depth)

    return node

def decompress_tree(node, parent_transform_chain=None):
    """
    Reconstructs points from the tree.
    parent_transform_chain: list of ellipsoids from root to current parent
    """
    if parent_transform_chain is None:
        parent_transform_chain = []

    current_chain = parent_transform_chain + [node.ellipsoid]

    if node.is_leaf:
        # 1. Reconstruct Local Points from Spherical
        local_pts = spherical_to_cartesian(
            node.residuals_r,
            node.residuals_theta,
            node.residuals_phi
        )

        # 2. Apply Inverse Chain (Leaf -> Root)
        # The points currently are in the local space of the Leaf Ellipsoid.
        # We need to walk UP the tree applying local_to_world.

        # Apply current leaf transform
        pts = local_to_world(local_pts, node.ellipsoid)

        # Apply parent transforms in reverse order (Parent of Leaf -> ... -> Root)
        # Note: We already applied the leaf's own transform above.
        # So we iterate through parent_transform_chain excluding the last one (current).
        for ellipsoid in reversed(parent_transform_chain[:-1]):
            pts = local_to_world(pts, ellipsoid)

        return pts

    else:
        # Internal Node: Recurse
        pts_left = decompress_tree(node.left, current_chain)
        pts_right = decompress_tree(node.right, current_chain)
        return np.concatenate([pts_left, pts_right], axis=0)

# --- Quantization Simulation & Metrics ---

def quantize_and_measure(node, r_bits=10, ang_bits=10):
    """
    Traverses tree, quantizes leaf residuals, calculates entropy.
    Returns: Total bits estimate, Quantized Node (copy)
    """
    if not node.is_leaf:
        bits_l, node_l = quantize_and_measure(node.left, r_bits, ang_bits)
        bits_r, node_r = quantize_and_measure(node.right, r_bits, ang_bits)

        # Copy structure
        new_node = CompressionNode(node.depth)
        new_node.ellipsoid = node.ellipsoid
        new_node.left = node_l
        new_node.right = node_r

        # Overhead: 10 floats for Ellipsoid (Center(3)+Radii(3)+Quat(4)) * 32 bits
        node_overhead = 10 * 32
        return bits_l + bits_r + node_overhead, new_node

    else:
        new_node = CompressionNode(node.depth)
        new_node.ellipsoid = node.ellipsoid
        new_node.is_leaf = True

        # --- Quantization Logic ---
        count = len(node.residuals_r)

        # Angular: Uniform Quantization [0, PI] and [0, 2PI]
        q_theta = np.round(node.residuals_theta / np.pi * (2**ang_bits - 1)).astype(int)
        q_phi   = np.round((node.residuals_phi + np.pi) / (2 * np.pi) * (2**ang_bits - 1)).astype(int)

        # Radial: Non-Uniform?
        # Ideally r is close to 1.0. We encode (r - 1.0).
        # Let's assume range [0.5, 1.5] covers most errors for now.
        r_min, r_max = 0.5, 1.5
        r_norm = (node.residuals_r - r_min) / (r_max - r_min)
        q_r = np.round(np.clip(r_norm, 0, 1) * (2**r_bits - 1)).astype(int)

        # --- Entropy Calculation ---
        def entropy(labels):
            if len(labels) == 0: return 0
            value, counts = np.unique(labels, return_counts=True)
            norm_counts = counts / counts.sum()
            return -np.sum(norm_counts * np.log2(norm_counts + 1e-9))

        ent_r = entropy(q_r)
        ent_theta = entropy(q_theta)
        ent_phi = entropy(q_phi)

        # Total bits = Entries * Entropy
        total_bits = count * (ent_r + ent_theta + ent_phi)

        # Dequantize for reconstruction check
        rec_theta = q_theta / (2**ang_bits - 1) * np.pi
        rec_phi   = q_phi / (2**ang_bits - 1) * (2 * np.pi) - np.pi
        rec_r     = (q_r / (2**r_bits - 1)) * (r_max - r_min) + r_min

        new_node.residuals_r = rec_r
        new_node.residuals_theta = rec_theta
        new_node.residuals_phi = rec_phi

        node_overhead = 10 * 32
        return total_bits + node_overhead, new_node

In [2]:
print("Generating 'Half-Ellipsoid' Point Cloud...")
# Generate random points on a half ellipsoid
num_points = 20000
u = np.random.uniform(0, 1, num_points)
v = np.random.uniform(0, 1, num_points)

# Spherical to Cartesian generator
theta = np.arccos(2 * u - 1) # Full sphere
phi = 2 * np.pi * v

# Restrict to half sphere (z > 0)
theta = theta[theta < np.pi/2]
phi = phi[:len(theta)]

# Ellipsoid parameters for generation
gt_radii = np.array([2.0, 1.0, 0.5])
x = gt_radii[0] * np.sin(theta) * np.cos(phi)
y = gt_radii[1] * np.sin(theta) * np.sin(phi)
z = gt_radii[2] * np.cos(theta)

# Add noise
points = np.stack([x, y, z], axis=1)
points += np.random.normal(0, 0.01, points.shape)
points = np.float32(points)

print(f"Original Data Size: {points.nbytes / 1024:.2f} KB (Float32)")

# 1. Compress (Build Tree)
DEPTH = 3
print(f"\nBuilding Tree (Depth {DEPTH})...")
root = build_tree(points, current_depth=0, max_depth=DEPTH)

# 2. Quantize & Estimate Size
print("Quantizing residuals (10 bits)...")
est_bits, quantized_root = quantize_and_measure(root, r_bits=10, ang_bits=10)
est_bytes = est_bits / 8
print(f"Estimated Compressed Size: {est_bytes / 1024:.2f} KB")
print(f"Compression Ratio: {points.nbytes / est_bytes:.2f}x")

# 3. Decompress
print("Decompressing...")
reconstructed_points = decompress_tree(quantized_root)

# 4. Validate
# Note: Points might be reordered due to splitting/sorting.
# To compare, we need to sort both arrays or use Nearest Neighbor.
# For a quick check, we assume the sorting logic in Decompression matches Compression
# (It DOES NOT automatically match if we just concat left+right, we lose the original index order).
# However, for visual/geometric checking, we care if the *set* of points matches.

from scipy.spatial import KDTree
kdtree = KDTree(points)
dists, _ = kdtree.query(reconstructed_points)
rmse = np.sqrt(np.mean(dists**2))

print(f"\nReconstruction RMSE: {rmse:.6f}")
print(f"Max Error: {np.max(dists):.6f}")

Generating 'Half-Ellipsoid' Point Cloud...
Original Data Size: 118.55 KB (Float32)

Building Tree (Depth 3)...
Quantizing residuals (10 bits)...
Estimated Compressed Size: 31.20 KB
Compression Ratio: 3.80x
Decompressing...

Reconstruction RMSE: 2.523198
Max Error: 5.163560


In [3]:
import numpy as np
from scipy.spatial.transform import Rotation as R
from scipy.optimize import least_squares

# --- Data Structures ---

class EllipsoidParams:
    def __init__(self, center, radii, rotation_matrix):
        self.center = center
        self.radii = radii
        self.rotation_matrix = rotation_matrix

    def get_transform_matrix(self):
        """
        Returns the 4x4 Affine Transform Matrix (Local -> World).
        T = Translate * Rotate * Scale
        """
        T = np.eye(4)
        T[:3, 3] = self.center

        Rot = np.eye(4)
        Rot[:3, :3] = self.rotation_matrix

        S = np.eye(4)
        S[0,0] = self.radii[0]
        S[1,1] = self.radii[1]
        S[2,2] = self.radii[2]

        # M = T * R * S
        return T @ Rot @ S

class CompressionNode:
    def __init__(self, depth):
        self.depth = depth
        self.ellipsoid = None
        self.left = None
        self.right = None
        self.is_leaf = False

        # Leaf Data
        self.residuals_r = None
        self.residuals_theta = None
        self.residuals_phi = None

        # Quantization Metadata (Calculated later)
        self.min_r = 0.0
        self.max_r = 0.0
        self.bits_r = 0
        self.bits_theta = 0
        self.bits_phi = 0

# --- Geometry & Fitting ---

def fit_ellipsoid(points):
    if len(points) < 4:
        return EllipsoidParams(np.mean(points, axis=0), np.ones(3)*0.001, np.eye(3))

    # 1. PCA Init
    center = np.mean(points, axis=0)
    centered = points - center
    cov = np.cov(centered, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)

    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    radii = 2.0 * np.sqrt(np.maximum(eigvals, 1e-6))
    rotation = eigvecs

    # 2. Optimization
    def transform_to_unit(p, c, r_log, rot_vec):
        rot_mat = R.from_rotvec(rot_vec).as_matrix()
        radii_val = np.exp(r_log)
        p_centered = p - c
        p_rotated = p_centered @ rot_mat
        p_scaled = p_rotated / radii_val
        return p_scaled

    def loss_function(params, p_target):
        c = params[0:3]
        r_log = params[3:6]
        rot_vec = params[6:9]
        p_unit = transform_to_unit(p_target, c, r_log, rot_vec)
        return np.linalg.norm(p_unit, axis=1) - 1.0

    r_vec_init = R.from_matrix(rotation).as_rotvec()
    x0 = np.concatenate([center, np.log(radii), r_vec_init])

    try:
        res = least_squares(loss_function, x0, args=(points,), method='lm', max_nfev=20)
        opt_c = res.x[0:3]
        opt_radii = np.exp(res.x[3:6])
        opt_rot = R.from_rotvec(res.x[6:9]).as_matrix()
        return EllipsoidParams(opt_c, opt_radii, opt_rot)
    except:
        return EllipsoidParams(center, radii, rotation)

def world_to_local(points, ellipsoid):
    p = points - ellipsoid.center
    p = p @ ellipsoid.rotation_matrix
    p = p / ellipsoid.radii
    return p

def cartesian_to_spherical(xyz):
    r = np.linalg.norm(xyz, axis=1)
    # Clip for safety
    theta = np.arccos(np.clip(xyz[:, 2] / (r + 1e-9), -1.0, 1.0))
    phi = np.arctan2(xyz[:, 1], xyz[:, 0])
    return r, theta, phi

def spherical_to_cartesian(r, theta, phi):
    x = r * np.sin(theta) * np.cos(phi)
    y = r * np.sin(theta) * np.sin(phi)
    z = r * np.cos(theta)
    return np.stack([x, y, z], axis=1)

# --- Tree Building ---

def build_tree(points, current_depth, max_depth):
    node = CompressionNode(current_depth)
    node.ellipsoid = fit_ellipsoid(points)
    local_points = world_to_local(points, node.ellipsoid)

    if current_depth >= max_depth or len(points) <= 4:
        node.is_leaf = True
        r, theta, phi = cartesian_to_spherical(local_points)
        node.residuals_r = r
        node.residuals_theta = theta
        node.residuals_phi = phi

        # Calculate precise ranges immediately for the optimizer to use
        node.min_r = np.min(r)
        node.max_r = np.max(r)
        return node

    # Split logic (Sort by X in local space)
    split_axis = 0
    sort_indices = np.argsort(local_points[:, split_axis])
    sorted_points = points[sort_indices]
    mid = len(sorted_points) // 2

    node.left = build_tree(sorted_points[:mid], current_depth + 1, max_depth)
    node.right = build_tree(sorted_points[mid:], current_depth + 1, max_depth)

    return node

# --- Adaptive Optimization Logic ---

def optimize_quantization(node, max_error_world, parent_transform=np.eye(4)):
    """
    Traverses the tree to calculate the required bits for each leaf.
    """
    # Calculate Cumulative Transform Matrix (M) for this node
    # M_current = M_parent * M_node
    # This matrix converts Local Node Space -> World Space
    local_transform = node.ellipsoid.get_transform_matrix()
    cumulative_transform = parent_transform @ local_transform

    if node.is_leaf:
        # 1. Calculate Maximum Expansion Factor
        # The singular values of the 3x3 linear part tell us the max stretch.
        linear_part = cumulative_transform[:3, :3]
        # Spectral norm = max singular value = max scaling factor
        max_scale_factor = np.linalg.norm(linear_part, ord=2)

        # 2. Determine Target Local Error
        # We want: local_error * max_scale_factor < max_error_world
        target_local_error = max_error_world / (max_scale_factor + 1e-9)

        # We distribute error budget.
        # Total Error^2 = dr^2 + (r*dtheta)^2 + (r*sin_theta*dphi)^2
        # Since r approx 1, and we want total error < target,
        # let's set per-component max error to target / sqrt(3).
        component_epsilon = target_local_error / np.sqrt(3)

        # 3. Calculate Bits for R
        r_range = node.max_r - node.min_r
        if r_range < 1e-9:
            node.bits_r = 0
        else:
            # step_size = range / (2^bits - 1)
            # We need step_size < component_epsilon
            # 2^bits > range / epsilon
            needed_steps = r_range / component_epsilon
            node.bits_r = int(np.ceil(np.log2(needed_steps + 1)))
            node.bits_r = max(1, min(node.bits_r, 32)) # Clamp

        # 4. Calculate Bits for Theta [0, PI]
        theta_range = np.pi
        needed_steps_theta = theta_range / component_epsilon
        node.bits_theta = int(np.ceil(np.log2(needed_steps_theta + 1)))
        node.bits_theta = max(1, min(node.bits_theta, 32))

        # 5. Calculate Bits for Phi [0, 2PI] (or actual range)
        # Using full range [0, 2pi] is safer unless we store min/max phi
        phi_range = 2 * np.pi
        needed_steps_phi = phi_range / component_epsilon
        node.bits_phi = int(np.ceil(np.log2(needed_steps_phi + 1)))
        node.bits_phi = max(1, min(node.bits_phi, 32))

        return

    # Recurse
    if node.left: optimize_quantization(node.left, max_error_world, cumulative_transform)
    if node.right: optimize_quantization(node.right, max_error_world, cumulative_transform)

# --- Quantization & Reconstruction ---

def get_quantized_data_size(node):
    """ Returns size in bits recursively """
    if node.is_leaf:
        count = len(node.residuals_r)
        # Metadata: min_r(32) + max_r(32) + bits_counts(3*8)
        meta_bits = 32 + 32 + 24
        payload_bits = count * (node.bits_r + node.bits_theta + node.bits_phi)
        return meta_bits + payload_bits + (10 * 32) # Ellipsoid params
    else:
        s_left = get_quantized_data_size(node.left) if node.left else 0
        s_right = get_quantized_data_size(node.right) if node.right else 0
        return s_left + s_right + (10 * 32) # Ellipsoid params

def decompress_recursive(node, parent_chain=None):
    if parent_chain is None: parent_chain = []

    if node.is_leaf:
        # 1. Dequantize (Simulated)
        # We simulate the loss by rounding to the calculated bit depth
        def quant_dequant(val, v_min, v_max, bits):
            if bits == 0: return np.full_like(val, v_min)
            steps = 2**bits - 1
            norm = (val - v_min) / (v_max - v_min)
            q = np.round(np.clip(norm, 0, 1) * steps)
            return (q / steps) * (v_max - v_min) + v_min

        r_rec = quant_dequant(node.residuals_r, node.min_r, node.max_r, node.bits_r)
        theta_rec = quant_dequant(node.residuals_theta, 0, np.pi, node.bits_theta)
        phi_rec = quant_dequant(node.residuals_phi, -np.pi, np.pi, node.bits_phi) # using -pi, pi for arctan2

        local = spherical_to_cartesian(r_rec, theta_rec, phi_rec)

        # 2. Transform to World
        # Apply leaf transform
        pts = (local * node.ellipsoid.radii) @ node.ellipsoid.rotation_matrix.T + node.ellipsoid.center

        # Apply parents
        for ell in reversed(parent_chain):
            pts = (pts * ell.radii) @ ell.rotation_matrix.T + ell.center
        return pts

    else:
        new_chain = parent_chain + [node.ellipsoid]
        pts_l = decompress_recursive(node.left, new_chain) if node.left else np.empty((0,3))
        pts_r = decompress_recursive(node.right, new_chain) if node.right else np.empty((0,3))
        return np.concatenate([pts_l, pts_r], axis=0)

In [4]:

print("Generating Test Data...")
num_points = 5000
# Generate a noisy sphere-like blob
vecs = np.random.randn(num_points, 3)
vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)
points = vecs * 10.0 + np.random.randn(num_points, 3) * 0.1

# Calculate AABB Diagonal
aabb_min = np.min(points, axis=0)
aabb_max = np.max(points, axis=0)
diagonal = np.linalg.norm(aabb_max - aabb_min)

# Define User Constraint
USER_ERROR_PERCENT = 0.01 # 0.01%
MAX_ERROR_WORLD = diagonal * (USER_ERROR_PERCENT / 100.0)

print(f"Data Diagonal: {diagonal:.4f}")
print(f"Max Allowed Error: {MAX_ERROR_WORLD:.6f} ({USER_ERROR_PERCENT}%)")

# 1. Build
print("\nBuilding Tree...")
root = build_tree(points, 0, max_depth=4)

# 2. Optimize Bits
print("Optimizing Bit Depths...")
optimize_quantization(root, MAX_ERROR_WORLD)

# 3. Analyze Leaf Bits
def print_leaf_stats(node):
    if node.is_leaf:
        print(f"Leaf D{node.depth}: Range R [{node.min_r:.3f}-{node.max_r:.3f}] -> Bits: R={node.bits_r}, Th={node.bits_theta}, Ph={node.bits_phi}")
    else:
        if node.left: print_leaf_stats(node.left)
        if node.right: print_leaf_stats(node.right)

print("\n--- Leaf Bit Allocations ---")
print_leaf_stats(root)

# 4. Measure Size
total_bits = get_quantized_data_size(root)
original_bits = num_points * 3 * 64 # Float64 input
print(f"\nOriginal Size: {original_bits/8/1024:.2f} KB")
print(f"Compressed Size: {total_bits/8/1024:.2f} KB")
print(f"Ratio: {original_bits/total_bits:.2f}x")

# 5. Verify Error
print("\nVerifying Reconstruction...")
rec_points = decompress_recursive(root)

# Simple nearest neighbor check (since order is permuted)
from scipy.spatial import KDTree
tree = KDTree(points)
dists, _ = tree.query(rec_points)

max_rec_error = np.max(dists)
print(f"Actual Max Error: {max_rec_error:.6f}")

if max_rec_error <= MAX_ERROR_WORLD:
    print("SUCCESS: Error is within bounds.")
else:
    print(f"WARNING: Error exceeded bounds by {max_rec_error - MAX_ERROR_WORLD:.6f}")

Generating Test Data...
Data Diagonal: 35.3182
Max Allowed Error: 0.003532 (0.01%)

Building Tree...
Optimizing Bit Depths...

--- Leaf Bit Allocations ---
Leaf D4: Range R [0.630-3.039] -> Bits: R=29, Th=29, Ph=30
Leaf D4: Range R [1.221-2.379] -> Bits: R=23, Th=24, Ph=25
Leaf D4: Range R [0.904-2.384] -> Bits: R=25, Th=26, Ph=27
Leaf D4: Range R [0.694-1.464] -> Bits: R=23, Th=25, Ph=26
Leaf D4: Range R [0.282-1.833] -> Bits: R=24, Th=25, Ph=26
Leaf D4: Range R [0.730-1.741] -> Bits: R=24, Th=26, Ph=27
Leaf D4: Range R [0.565-2.262] -> Bits: R=26, Th=27, Ph=28
Leaf D4: Range R [0.783-1.692] -> Bits: R=24, Th=25, Ph=26
Leaf D4: Range R [0.022-4.176] -> Bits: R=32, Th=32, Ph=32
Leaf D4: Range R [1.034-1.526] -> Bits: R=24, Th=26, Ph=27
Leaf D4: Range R [1.415-2.567] -> Bits: R=29, Th=30, Ph=31
Leaf D4: Range R [1.090-1.741] -> Bits: R=26, Th=29, Ph=30
Leaf D4: Range R [0.758-2.046] -> Bits: R=24, Th=25, Ph=26
Leaf D4: Range R [0.807-1.507] -> Bits: R=23, Th=25, Ph=26
Leaf D4: Range R [

In [5]:
import numpy as np
from scipy.spatial.transform import Rotation as R
from scipy.optimize import least_squares

# --- Data Structures ---

class EllipsoidParams:
    def __init__(self, center, radii, rotation_matrix):
        self.center = center
        self.radii = radii
        self.rotation_matrix = rotation_matrix

class CompressionNode:
    def __init__(self, depth):
        self.depth = depth
        self.ellipsoid = None
        self.left = None
        self.right = None
        self.is_leaf = False

        # Leaf Data
        self.residuals_r = None
        self.residuals_theta = None
        self.residuals_phi = None

        self.min_r, self.max_r = 0.0, 0.0
        self.bits_r, self.bits_theta, self.bits_phi = 0, 0, 0

# --- Geometry Functions ---

def fit_ellipsoid_to_local(points):
    if len(points) < 4:
        return EllipsoidParams(np.mean(points, axis=0), np.ones(3), np.eye(3))

    # 1. PCA Init
    center = np.mean(points, axis=0)
    centered = points - center
    cov = np.cov(centered, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)

    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    radii = 2.0 * np.sqrt(np.maximum(eigvals, 1e-6))
    rotation = eigvecs

    # 2. Optimization
    def get_unit_error(params, p_target):
        c = params[0:3]
        r_log = params[3:6]
        rot_vec = params[6:9]

        rot_mat = R.from_rotvec(rot_vec).as_matrix()
        radii_val = np.exp(r_log)

        # Transform: R.T * (P - C) / S
        p_centered = p_target - c
        p_rotated = p_centered @ rot_mat
        p_scaled = p_rotated / radii_val

        return np.linalg.norm(p_scaled, axis=1) - 1.0

    r_vec_init = R.from_matrix(rotation).as_rotvec()
    x0 = np.concatenate([center, np.log(radii), r_vec_init])

    try:
        res = least_squares(get_unit_error, x0, args=(points,), method='lm', max_nfev=10)
        opt_c = res.x[0:3]
        opt_radii = np.exp(res.x[3:6])
        opt_rot = R.from_rotvec(res.x[6:9]).as_matrix()
        return EllipsoidParams(opt_c, opt_radii, opt_rot)
    except:
        return EllipsoidParams(center, radii, rotation)

def transform_forward(points, ellipsoid):
    """ World -> Local (Node i -> Node i+1) """
    p = points - ellipsoid.center
    p = p @ ellipsoid.rotation_matrix
    p = p / ellipsoid.radii
    return p

def transform_inverse(local_points, ellipsoid):
    """ Local -> World (Node i+1 -> Node i) """
    p = local_points * ellipsoid.radii
    p = p @ ellipsoid.rotation_matrix.T
    p = p + ellipsoid.center
    return p

# --- Tree Logic (Order Preserving) ---

def build_tree_recursive(points, current_depth, max_depth):
    node = CompressionNode(current_depth)

    # 1. Fit Ellipsoid to CURRENT points
    node.ellipsoid = fit_ellipsoid_to_local(points)

    # 2. Transform points into THIS ellipsoid's local space
    next_points = transform_forward(points, node.ellipsoid)

    # 3. Check Termination
    if current_depth >= max_depth or len(points) <= 4:
        node.is_leaf = True

        r = np.linalg.norm(next_points, axis=1)
        z_div_r = next_points[:, 2] / (r + 1e-9)
        z_div_r = np.clip(z_div_r, -1.0, 1.0)
        theta = np.arccos(z_div_r)
        phi = np.arctan2(next_points[:, 1], next_points[:, 0])

        node.residuals_r = r
        node.residuals_theta = theta
        node.residuals_phi = phi

        node.min_r, node.max_r = np.min(r), np.max(r)
        return node

    # 4. Simple Split (Preserves Order)
    # We just cut the array in half based on index.
    mid = len(next_points) // 2

    # Since we are not sorting, next_points[:mid] corresponds exactly
    # to points[:mid] in the parent's frame.
    node.left = build_tree_recursive(next_points[:mid], current_depth + 1, max_depth)
    node.right = build_tree_recursive(next_points[mid:], current_depth + 1, max_depth)

    return node

# --- Decompression ---

def decompress_recursive(node):
    if node.is_leaf:
        # Simulate Dequantization
        # Note: In a real implementation, you'd use node.bits_r/theta/phi here
        local_x = node.residuals_r * np.sin(node.residuals_theta) * np.cos(node.residuals_phi)
        local_y = node.residuals_r * np.sin(node.residuals_theta) * np.sin(node.residuals_phi)
        local_z = node.residuals_r * np.cos(node.residuals_theta)
        local_points = np.stack([local_x, local_y, local_z], axis=1)

        # Transform Local -> Parent
        return transform_inverse(local_points, node.ellipsoid)

    else:
        # Recurse
        # Since we split by simple slicing [:mid], [mid:],
        # we reconstruct by simple concatenation [left, right]
        pts_left = decompress_recursive(node.left) if node.left else np.empty((0,3))
        pts_right = decompress_recursive(node.right) if node.right else np.empty((0,3))

        combined_pts_local = np.concatenate([pts_left, pts_right], axis=0)

        # Transform Children -> Parent
        return transform_inverse(combined_pts_local, node.ellipsoid)

# --- Bit Optimization ---

def optimize_bits(node, max_error_world, parent_scale_factor=1.0):
    current_expansion = np.max(node.ellipsoid.radii)
    total_scale = parent_scale_factor * current_expansion

    if node.is_leaf:
        target_local_err = max_error_world / (total_scale + 1e-9)

        r_range = node.max_r - node.min_r
        if r_range < 1e-9: r_range = 1e-6

        # Bits for R
        steps = r_range / target_local_err
        node.bits_r = int(np.ceil(np.log2(steps + 1)))
        node.bits_r = np.clip(node.bits_r, 4, 32)

        # Bits for Angles
        circ = 2 * np.pi
        steps_ang = circ / target_local_err
        node.bits_theta = int(np.ceil(np.log2(steps_ang + 1)))
        node.bits_theta = np.clip(node.bits_theta, 4, 32)
        node.bits_phi = node.bits_theta
    else:
        if node.left: optimize_bits(node.left, max_error_world, total_scale)
        if node.right: optimize_bits(node.right, max_error_world, total_scale)

def get_total_size(node):
    size = 320 # Ellipsoid overhead (10 floats * 32)
    if node.is_leaf:
        count = len(node.residuals_r)
        size += count * (node.bits_r + node.bits_theta + node.bits_phi)
        size += 64 + 24 # Metadata
    else:
        if node.left: size += get_total_size(node.left)
        if node.right: size += get_total_size(node.right)
    return size

In [6]:
print("Generating Test Data...")
num_points = 50000
vecs = np.random.randn(num_points, 3)
vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)
points = vecs * 10.0 + np.random.randn(num_points, 3) * 0.05

diagonal = np.linalg.norm(np.max(points, axis=0) - np.min(points, axis=0))
MAX_ERROR = diagonal * 0.0001
print(f"Target Max Error: {MAX_ERROR:.6f}")

print("\n1. Building Recursive Tree...")
root = build_tree_recursive(points, 0, max_depth=8)

print("2. Optimizing Bits...")
optimize_bits(root, MAX_ERROR)

print("3. Measuring Size...")
bits = get_total_size(root)
bytes_val = bits / 8
print(f"Compressed: {bytes_val/1024:.2f} KB")
print(f"Original: {(num_points*12)/1024:.2f} KB")
print(f"Ratio: {(num_points*12)/bytes_val:.2f}x")

print("\n4. Decompressing...")
rec_points = decompress_recursive(root)

# Direct Verification (Index to Index)
# Because we didn't sort, rec_points[i] corresponds to points[i]
diff = points - rec_points
dist = np.linalg.norm(diff, axis=1)
max_rec_error = np.max(dist)

print(f"Max Reconstruction Error: {max_rec_error:.6f}")

if max_rec_error < MAX_ERROR * 2.0: # Slight margin for optimization noise
    print("Result: SUCCESS (Order Preserved)")
else:
    print("Result: ERROR EXCEEDED")

Generating Test Data...
Target Max Error: 0.003502

1. Building Recursive Tree...
2. Optimizing Bits...
3. Measuring Size...
Compressed: 252.55 KB
Original: 585.94 KB
Ratio: 2.32x

4. Decompressing...
Max Reconstruction Error: 0.000001
Result: SUCCESS (Order Preserved)


In [7]:
import numpy as np
from scipy.spatial.transform import Rotation as R
from scipy.optimize import least_squares
from collections import namedtuple

# --- Data Structures ---

class EllipsoidParams:
    def __init__(self, center, radii, rotation_matrix):
        self.center = center
        self.radii = radii
        self.rotation_matrix = rotation_matrix

class CompressionNode:
    def __init__(self, depth):
        self.depth = depth
        self.ellipsoid = None
        self.left = None
        self.right = None
        self.is_leaf = False

        # Leaf Data (Floats)
        self.residuals_r = None
        self.residuals_theta = None
        self.residuals_phi = None

        # Metadata
        self.min_r, self.max_r = 0.0, 0.0
        self.bits_r, self.bits_theta, self.bits_phi = 0, 0, 0

        # Entropy Stats
        self.entropy_r = 0.0
        self.entropy_theta = 0.0
        self.entropy_phi = 0.0

Stats = namedtuple('Stats', ['header_bits', 'payload_raw_bits', 'payload_entropy_bits'])

# --- Geometry Functions ---

def fit_ellipsoid_to_local(points):
    if len(points) < 4:
        return EllipsoidParams(np.mean(points, axis=0), np.ones(3), np.eye(3))

    # 1. PCA Init
    center = np.mean(points, axis=0)
    centered = points - center
    cov = np.cov(centered, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)

    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    radii = 2.0 * np.sqrt(np.maximum(eigvals, 1e-6))
    rotation = eigvecs

    # 2. Optimization
    def get_unit_error(params, p_target):
        c = params[0:3]
        r_log = params[3:6]
        rot_vec = params[6:9]
        rot_mat = R.from_rotvec(rot_vec).as_matrix()
        radii_val = np.exp(r_log)
        p_centered = p_target - c
        p_rotated = p_centered @ rot_mat
        p_scaled = p_rotated / radii_val
        return np.linalg.norm(p_scaled, axis=1) - 1.0

    r_vec_init = R.from_matrix(rotation).as_rotvec()
    x0 = np.concatenate([center, np.log(radii), r_vec_init])

    try:
        res = least_squares(get_unit_error, x0, args=(points,), method='lm', max_nfev=50)
        opt_c = res.x[0:3]
        opt_radii = np.exp(res.x[3:6])
        opt_rot = R.from_rotvec(res.x[6:9]).as_matrix()
        return EllipsoidParams(opt_c, opt_radii, opt_rot)
    except:
        return EllipsoidParams(center, radii, rotation)

def transform_forward(points, ellipsoid):
    p = points - ellipsoid.center
    p = p @ ellipsoid.rotation_matrix
    p = p / ellipsoid.radii
    return p

def transform_inverse(local_points, ellipsoid):
    p = local_points * ellipsoid.radii
    p = p @ ellipsoid.rotation_matrix.T
    p = p + ellipsoid.center
    return p

# --- Tree Logic (Order Preserving) ---

def build_tree_recursive(points, current_depth, max_depth):
    node = CompressionNode(current_depth)

    node.ellipsoid = fit_ellipsoid_to_local(points)
    next_points = transform_forward(points, node.ellipsoid)

    if current_depth >= max_depth or len(points) <= 4:
        node.is_leaf = True
        r = np.linalg.norm(next_points, axis=1)
        z_div_r = np.clip(next_points[:, 2] / (r + 1e-9), -1.0, 1.0)
        theta = np.arccos(z_div_r)
        phi = np.arctan2(next_points[:, 1], next_points[:, 0])

        node.residuals_r = r
        node.residuals_theta = theta
        node.residuals_phi = phi
        node.min_r, node.max_r = np.min(r), np.max(r)
        return node

    mid = len(next_points) // 2
    node.left = build_tree_recursive(next_points[:mid], current_depth + 1, max_depth)
    node.right = build_tree_recursive(next_points[mid:], current_depth + 1, max_depth)
    return node

# --- Decompression ---

def decompress_recursive(node):
    if node.is_leaf:
        local_x = node.residuals_r * np.sin(node.residuals_theta) * np.cos(node.residuals_phi)
        local_y = node.residuals_r * np.sin(node.residuals_theta) * np.sin(node.residuals_phi)
        local_z = node.residuals_r * np.cos(node.residuals_theta)
        local_points = np.stack([local_x, local_y, local_z], axis=1)
        return transform_inverse(local_points, node.ellipsoid)
    else:
        pts_left = decompress_recursive(node.left) if node.left else np.empty((0,3))
        pts_right = decompress_recursive(node.right) if node.right else np.empty((0,3))
        combined_pts_local = np.concatenate([pts_left, pts_right], axis=0)
        return transform_inverse(combined_pts_local, node.ellipsoid)

# --- Analysis & Quantization ---

def calculate_entropy(indices):
    if len(indices) == 0: return 0.0
    _, counts = np.unique(indices, return_counts=True)
    probs = counts / len(indices)
    return -np.sum(probs * np.log2(probs + 1e-9))

def analyze_and_quantize(node, max_error_world, parent_scale_factor=1.0):
    """
    1. Propagates scale factor.
    2. Calculates required bits.
    3. Simulates quantization.
    4. Calculates entropy of the quantized stream.
    """
    current_expansion = np.max(node.ellipsoid.radii)
    total_scale = parent_scale_factor * current_expansion

    # Header Cost: 10 floats (C,R,Q) + 1 byte flags
    node_header_bits = (10 * 32) + 8

    if node.is_leaf:
        target_local_err = max_error_world / (total_scale + 1e-9)

        # --- R Quantization ---
        r_range = node.max_r - node.min_r
        if r_range < 1e-9: r_range = 1e-6
        steps_r = r_range / target_local_err
        node.bits_r = int(np.clip(np.ceil(np.log2(steps_r + 1)), 4, 32))

        # Quantize R
        q_r = np.round((node.residuals_r - node.min_r) / r_range * (2**node.bits_r - 1)).astype(int)
        node.entropy_r = calculate_entropy(q_r)

        # --- Angle Quantization ---
        # Heuristic: Angles cover 2PI.
        steps_ang = (2 * np.pi) / target_local_err
        node.bits_theta = int(np.clip(np.ceil(np.log2(steps_ang + 1)), 4, 32))
        node.bits_phi = node.bits_theta

        # Quantize Theta [0, PI]
        q_theta = np.round(node.residuals_theta / np.pi * (2**node.bits_theta - 1)).astype(int)
        node.entropy_theta = calculate_entropy(q_theta)

        # Quantize Phi [-PI, PI]
        q_phi = np.round((node.residuals_phi + np.pi) / (2 * np.pi) * (2**node.bits_phi - 1)).astype(int)
        node.entropy_phi = calculate_entropy(q_phi)

        count = len(node.residuals_r)

        # Payload: Raw Bits
        payload_raw = count * (node.bits_r + node.bits_theta + node.bits_phi)

        # Payload: Entropy Bits (Theoretical Limit)
        payload_entropy = count * (node.entropy_r + node.entropy_theta + node.entropy_phi)

        # Metadata (min_r, max_r, bit_counts)
        meta_bits = 64 + 24

        return Stats(node_header_bits + meta_bits, payload_raw, payload_entropy)

    else:
        stats_left = analyze_and_quantize(node.left, max_error_world, total_scale) if node.left else Stats(0,0,0)
        stats_right = analyze_and_quantize(node.right, max_error_world, total_scale) if node.right else Stats(0,0,0)

        return Stats(
            node_header_bits + stats_left.header_bits + stats_right.header_bits,
            stats_left.payload_raw_bits + stats_right.payload_raw_bits,
            stats_left.payload_entropy_bits + stats_right.payload_entropy_bits
        )

In [8]:
np.random.seed(42)
print("Generating Test Data...")
num_points = 100000
vecs = np.random.randn(num_points, 3)
vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)
# Create a noisy ellipsoid
points = vecs * np.array([10.0, 5.0, 2.0]) + np.random.randn(num_points, 3) * 0.05

diagonal = np.linalg.norm(np.max(points, axis=0) - np.min(points, axis=0))
MAX_ERROR = diagonal * 0.0001 # 0.01%
print(f"Count: {num_points}")
print(f"Target Max Error: {MAX_ERROR:.6f}")

Generating Test Data...
Count: 100000
Target Max Error: 0.002307


In [9]:
np.random.seed(42)
print("\n1. Building Recursive Tree (Depth 5)...")
root = build_tree_recursive(points, 0, max_depth=5)

print("2. Analyzing Entropy & Quantization...")
stats = analyze_and_quantize(root, MAX_ERROR)

# Calculations
orig_bytes = num_points * 3 * 4 # Double precision input (12 bytes per point)

# Raw Output (Bitstream without Entropy Coding)
total_raw_bits = stats.header_bits + stats.payload_raw_bits
total_raw_bytes = total_raw_bits / 8

# Entropy Output (Theoretical Limit with Arithmetic Coding)
total_ent_bits = stats.header_bits + stats.payload_entropy_bits
total_ent_bytes = total_ent_bits / 8

print("\n--- Detailed Results ---")
print(f"Original Size (Float32):  {orig_bytes/1024:.2f} KB")
print("-" * 30)
print(f"Header Overhead:          {stats.header_bits/8/1024:.2f} KB ({(stats.header_bits/total_ent_bits)*100:.1f}%)")
print(f"Data Payload (Raw Bits):  {stats.payload_raw_bits/8/1024:.2f} KB")
print(f"Data Payload (Entropy):   {stats.payload_entropy_bits/8/1024:.2f} KB")
print("-" * 30)
print(f"Total Size (Standard):    {total_raw_bytes/1024:.2f} KB (Ratio: {orig_bytes/total_raw_bytes:.2f}x)")
print(f"Total Size (Ent. Coded):  {total_ent_bytes/1024:.2f} KB (Ratio: {orig_bytes/total_ent_bytes:.2f}x)")

# Verification
print("\n3. Verifying Reconstruction...")
rec_points = decompress_recursive(root)
diff = points - rec_points
dist = np.linalg.norm(diff, axis=1)
max_rec_error = np.max(dist)

print(f"Max Reconstruction Error: {max_rec_error:.6f}")

if max_rec_error < MAX_ERROR * 2.0:
    print("Status: SUCCESS")
else:
    print("Status: FAILED (Error too high)")


1. Building Recursive Tree (Depth 5)...
2. Analyzing Entropy & Quantization...

--- Detailed Results ---
Original Size (Float32):  1171.88 KB
------------------------------
Header Overhead:          2.87 KB (0.7%)
Data Payload (Raw Bits):  488.28 KB
Data Payload (Entropy):   386.09 KB
------------------------------
Total Size (Standard):    491.15 KB (Ratio: 2.39x)
Total Size (Ent. Coded):  388.96 KB (Ratio: 3.01x)

3. Verifying Reconstruction...
Max Reconstruction Error: 0.000001
Status: SUCCESS


In [10]:
np.random.seed(42)
print("\n1. Building Recursive Tree (Depth 8)...")
root = build_tree_recursive(points, 0, max_depth=8)

print("2. Analyzing Entropy & Quantization...")
stats = analyze_and_quantize(root, MAX_ERROR)

# Calculations
orig_bytes = num_points * 3 * 4 # Double precision input (12 bytes per point)

# Raw Output (Bitstream without Entropy Coding)
total_raw_bits = stats.header_bits + stats.payload_raw_bits
total_raw_bytes = total_raw_bits / 8

# Entropy Output (Theoretical Limit with Arithmetic Coding)
total_ent_bits = stats.header_bits + stats.payload_entropy_bits
total_ent_bytes = total_ent_bits / 8

print("\n--- Detailed Results ---")
print(f"Original Size (Float32):  {orig_bytes/1024:.2f} KB")
print("-" * 30)
print(f"Header Overhead:          {stats.header_bits/8/1024:.2f} KB ({(stats.header_bits/total_ent_bits)*100:.1f}%)")
print(f"Data Payload (Raw Bits):  {stats.payload_raw_bits/8/1024:.2f} KB")
print(f"Data Payload (Entropy):   {stats.payload_entropy_bits/8/1024:.2f} KB")
print("-" * 30)
print(f"Total Size (Standard):    {total_raw_bytes/1024:.2f} KB (Ratio: {orig_bytes/total_raw_bytes:.2f}x)")
print(f"Total Size (Ent. Coded):  {total_ent_bytes/1024:.2f} KB (Ratio: {orig_bytes/total_ent_bytes:.2f}x)")

# Verification
print("\n3. Verifying Reconstruction...")
rec_points = decompress_recursive(root)
diff = points - rec_points
dist = np.linalg.norm(diff, axis=1)
max_rec_error = np.max(dist)

print(f"Max Reconstruction Error: {max_rec_error:.6f}")

if max_rec_error < MAX_ERROR * 2.0:
    print("Status: SUCCESS")
else:
    print("Status: FAILED (Error too high)")


1. Building Recursive Tree (Depth 8)...
2. Analyzing Entropy & Quantization...

--- Detailed Results ---
Original Size (Float32):  1171.88 KB
------------------------------
Header Overhead:          23.21 KB (7.1%)
Data Payload (Raw Bits):  480.08 KB
Data Payload (Entropy):   302.42 KB
------------------------------
Total Size (Standard):    503.29 KB (Ratio: 2.33x)
Total Size (Ent. Coded):  325.63 KB (Ratio: 3.60x)

3. Verifying Reconstruction...
Max Reconstruction Error: 0.000002
Status: SUCCESS


In [11]:
root.left.left.left.left.left.left.left.left.bits_r

9

In [12]:
root.left.left.left.left.left.left.left.left.bits_theta

15

In [13]:
root.left.left.left.left.left.left.left.left.bits_phi

15

In [14]:
import numpy as np
from scipy.spatial.transform import Rotation as R
from scipy.optimize import least_squares
from collections import namedtuple

# --- Data Structures ---

class EllipsoidParams:
    def __init__(self, center, radii, rotation_matrix):
        self.center = center
        self.radii = radii
        self.rotation_matrix = rotation_matrix

class CompressionNode:
    def __init__(self, depth):
        self.depth = depth
        self.ellipsoid = None
        self.left = None
        self.right = None
        self.is_leaf = False

        # Leaf Data (Floats)
        self.residuals_r = None
        self.residuals_theta = None
        self.residuals_phi = None

        # Metadata ranges
        self.min_r, self.max_r = 0.0, 0.0
        self.min_theta, self.max_theta = 0.0, 0.0
        self.min_phi, self.max_phi = 0.0, 0.0

        # Bit allocations
        self.bits_r, self.bits_theta, self.bits_phi = 0, 0, 0

        # Entropy Stats
        self.entropy_r = 0.0
        self.entropy_theta = 0.0
        self.entropy_phi = 0.0

Stats = namedtuple('Stats', ['header_bits', 'payload_raw_bits', 'payload_entropy_bits'])

# --- Geometry & Optimization Functions ---

def fit_ellipsoid_shape(points):
    """ Step 1: Optimize Center and Radii (Surface Fit) """
    if len(points) < 4:
        return EllipsoidParams(np.mean(points, axis=0), np.ones(3), np.eye(3))

    # PCA Init
    center = np.mean(points, axis=0)
    centered = points - center
    cov = np.cov(centered, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    idx = np.argsort(eigvals)[::-1]
    radii = 2.0 * np.sqrt(np.maximum(eigvals[idx], 1e-6))
    rotation = eigvecs[:, idx]

    # Optimization (Minimize Radial Error)
    def get_unit_error(params, p_target):
        c = params[0:3]
        r_log = params[3:6]
        rot_vec = params[6:9]
        rot_mat = R.from_rotvec(rot_vec).as_matrix()
        radii_val = np.exp(r_log)
        # R.T * (P - C) / S
        p_centered = p_target - c
        p_rotated = p_centered @ rot_mat
        p_scaled = p_rotated / radii_val
        return np.linalg.norm(p_scaled, axis=1) - 1.0

    x0 = np.concatenate([center, np.log(radii), R.from_matrix(rotation).as_rotvec()])

    try:
        res = least_squares(get_unit_error, x0, args=(points,), method='lm', max_nfev=10)
        opt_c = res.x[0:3]
        opt_radii = np.exp(res.x[3:6])
        opt_rot = R.from_rotvec(res.x[6:9]).as_matrix()
        return EllipsoidParams(opt_c, opt_radii, opt_rot)
    except:
        return EllipsoidParams(center, radii, rotation)

def align_ellipsoid_grid(points, ellipsoid):
    """
    Step 2: "Group" Optimization.
    Rotate the ellipsoid (without changing shape) so residuals are aligned
    with coordinate axes, minimizing the Theta/Phi bounding box.
    """
    # 1. Transform points to current local space
    p = points - ellipsoid.center
    p = p @ ellipsoid.rotation_matrix
    p = p / ellipsoid.radii

    # 2. Compute PCA of the LOCAL points
    # We want to find a rotation R_local that aligns these points
    cov = np.cov(p, rowvar=False)
    _, eigvecs = np.linalg.eigh(cov)

    # eigvecs is the rotation needed to align local points to axes
    # We want to bake this into the Ellipsoid's rotation.
    # New Local = Old Local @ Eigvecs
    # So we need to Rotate the Ellipsoid by Eigvecs.

    # Update Ellipsoid Rotation: R_new = R_old @ Eigvecs
    new_rot = ellipsoid.rotation_matrix @ eigvecs

    return EllipsoidParams(ellipsoid.center, ellipsoid.radii, new_rot)

def transform_forward(points, ellipsoid):
    p = points - ellipsoid.center
    p = p @ ellipsoid.rotation_matrix
    p = p / ellipsoid.radii
    return p

def transform_inverse(local_points, ellipsoid):
    p = local_points * ellipsoid.radii
    p = p @ ellipsoid.rotation_matrix.T
    p = p + ellipsoid.center
    return p

# --- Tree Building ---

def build_tree_recursive(points, current_depth, max_depth):
    node = CompressionNode(current_depth)

    # 1. Fit Shape (Minimize Radial Error)
    node.ellipsoid = fit_ellipsoid_shape(points)

    # 2. Align Grid (Minimize Angular Spread) - The "Grouping" optimization
    node.ellipsoid = align_ellipsoid_grid(points, node.ellipsoid)

    # 3. Transform
    next_points = transform_forward(points, node.ellipsoid)

    # 4. Leaf Check
    if current_depth >= max_depth or len(points) <= 4:
        node.is_leaf = True

        r = np.linalg.norm(next_points, axis=1)
        # Safe Arccos
        z_div_r = np.clip(next_points[:, 2] / (r + 1e-9), -1.0, 1.0)
        theta = np.arccos(z_div_r)
        phi = np.arctan2(next_points[:, 1], next_points[:, 0])

        node.residuals_r = r
        node.residuals_theta = theta
        node.residuals_phi = phi

        # Store Ranges (Crucial for low bits)
        node.min_r, node.max_r = np.min(r), np.max(r)
        node.min_theta, node.max_theta = np.min(theta), np.max(theta)
        node.min_phi, node.max_phi = np.min(phi), np.max(phi)

        return node

    # 5. Split (Spatial)
    # Since we aligned the grid, X-axis (index 0) is the principal axis!
    split_axis = 0
    sort_indices = np.argsort(next_points[:, split_axis])
    sorted_points = next_points[sort_indices]

    mid = len(sorted_points) // 2

    node.left = build_tree_recursive(sorted_points[:mid], current_depth + 1, max_depth)
    node.right = build_tree_recursive(sorted_points[mid:], current_depth + 1, max_depth)
    return node

# --- Analysis (Adaptive Bits) ---

def calculate_entropy(indices):
    if len(indices) == 0: return 0.0
    _, counts = np.unique(indices, return_counts=True)
    probs = counts / len(indices)
    return -np.sum(probs * np.log2(probs + 1e-9))

def analyze_and_quantize(node, max_error_world, parent_scale_factor=1.0):
    current_expansion = np.max(node.ellipsoid.radii)
    total_scale = parent_scale_factor * current_expansion

    # 10 floats (ellipsoid) + 6 floats (ranges) = 16 * 32 bits
    node_header_bits = (16 * 32)

    if node.is_leaf:
        target_local_err = max_error_world / (total_scale + 1e-9)

        # Helper for bits calculation
        def calc_bits(val_range, error_budget):
            if val_range < 1e-9: return 0 # No variance, 0 bits needed
            # We assume uniform quantization logic: step = range / (2^b - 1)
            # We need step < error_budget
            steps = val_range / error_budget
            if steps <= 1: return 1
            return int(np.clip(np.ceil(np.log2(steps)), 1, 32))

        # --- Radial Bits ---
        node.bits_r = calc_bits(node.max_r - node.min_r, target_local_err)

        # --- Angular Bits (Adaptive Range) ---
        # The error budget for angles (arc length) is approx radius * angle_error
        # Since r ~ 1.0, angle_error_budget ~ target_local_err
        node.bits_theta = calc_bits(node.max_theta - node.min_theta, target_local_err)
        node.bits_phi = calc_bits(node.max_phi - node.min_phi, target_local_err)

        # --- Entropy Calculation ---
        def quantize_sim(vals, vmin, vmax, bits):
            if bits == 0: return np.zeros(len(vals), dtype=int)
            rnge = vmax - vmin
            if rnge < 1e-9: return np.zeros(len(vals), dtype=int)
            norm = (vals - vmin) / rnge
            return np.round(np.clip(norm, 0, 1) * (2**bits - 1)).astype(int)

        q_r = quantize_sim(node.residuals_r, node.min_r, node.max_r, node.bits_r)
        q_theta = quantize_sim(node.residuals_theta, node.min_theta, node.max_theta, node.bits_theta)
        q_phi = quantize_sim(node.residuals_phi, node.min_phi, node.max_phi, node.bits_phi)

        node.entropy_r = calculate_entropy(q_r)
        node.entropy_theta = calculate_entropy(q_theta)
        node.entropy_phi = calculate_entropy(q_phi)

        count = len(node.residuals_r)
        payload_raw = count * (node.bits_r + node.bits_theta + node.bits_phi)
        payload_entropy = count * (node.entropy_r + node.entropy_theta + node.entropy_phi)

        return Stats(node_header_bits, payload_raw, payload_entropy)

    else:
        stats_left = analyze_and_quantize(node.left, max_error_world, total_scale) if node.left else Stats(0,0,0)
        stats_right = analyze_and_quantize(node.right, max_error_world, total_scale) if node.right else Stats(0,0,0)

        return Stats(
            node_header_bits + stats_left.header_bits + stats_right.header_bits,
            stats_left.payload_raw_bits + stats_right.payload_raw_bits,
            stats_left.payload_entropy_bits + stats_right.payload_entropy_bits
        )

# --- Main Test ---

if __name__ == "__main__":
    np.random.seed(42)
    print("Generating Test Data (Clustered Surface)...")
    num_points = 5000
    # Generate points that are NOT a full sphere, but a patch
    u = np.random.uniform(0.4, 0.6, num_points) # Small patch in theta
    v = np.random.uniform(0.1, 0.3, num_points) # Small patch in phi

    # Map to sphere
    theta = u * np.pi
    phi = v * 2 * np.pi
    x = np.sin(theta) * np.cos(phi)
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(theta)

    # Scale to ellipsoid
    points = np.stack([x, y, z], axis=1) * np.array([10.0, 5.0, 2.0])
    points += np.random.normal(0, 0.001, points.shape) # Tiny noise

    diagonal = np.linalg.norm(np.max(points, axis=0) - np.min(points, axis=0))
    MAX_ERROR = diagonal * 0.0001 # 0.01%
    print(f"Target Max Error: {MAX_ERROR:.6f}")

    print("\n1. Building Recursive Tree (Depth 8)...")
    root = build_tree_recursive(points, 0, max_depth=8)

    print("2. Analyzing...")
    stats = analyze_and_quantize(root, MAX_ERROR)

    def print_leaf_bits(node, limit=5):
        if limit <= 0: return limit
        if node.is_leaf:
            rng_t = node.max_theta - node.min_theta
            print(f"Leaf D{node.depth}: Theta Range {rng_t:.4f} rad -> Bits: {node.bits_theta}")
            return limit - 1
        l = print_leaf_bits(node.left, limit)
        return print_leaf_bits(node.right, l)

    print("\n--- Bit Allocation Check ---")
    print_leaf_bits(root)

    total_ent_bits = stats.header_bits + stats.payload_entropy_bits
    total_ent_bytes = total_ent_bits / 8
    orig_bytes = num_points * 3 * 8

    print("\n--- Results ---")
    print(f"Header Overhead: {stats.header_bits/8/1024:.2f} KB")
    print(f"Entropy Payload: {stats.payload_entropy_bits/8/1024:.2f} KB")
    print(f"Total Compressed: {total_ent_bytes/1024:.2f} KB")
    print(f"Original: {orig_bytes/1024:.2f} KB")
    print(f"Ratio: {orig_bytes/total_ent_bytes:.2f}x")

Generating Test Data (Clustered Surface)...
Target Max Error: 0.001144

1. Building Recursive Tree (Depth 8)...


D:\meshpress\.venv\Lib\site-packages\numpy\linalg\_linalg.py:2832: RuntimeWarning: overflow encountered in multiply
  s = (x.conj() * x).real
C:\Users\Denys\AppData\Local\Temp\ipykernel_22740\2824189438.py:68: RuntimeWarning: divide by zero encountered in divide
  p_scaled = p_rotated / radii_val
C:\Users\Denys\AppData\Local\Temp\ipykernel_22740\2824189438.py:64: RuntimeWarning: overflow encountered in exp
  radii_val = np.exp(r_log)
C:\Users\Denys\AppData\Local\Temp\ipykernel_22740\2824189438.py:76: RuntimeWarning: overflow encountered in exp
  opt_radii = np.exp(res.x[3:6])


2. Analyzing...

--- Bit Allocation Check ---
Leaf D8: Theta Range 3.1416 rad -> Bits: 32
Leaf D8: Theta Range 0.0000 rad -> Bits: 32
Leaf D8: Theta Range 2.1284 rad -> Bits: 32
Leaf D8: Theta Range 3.1416 rad -> Bits: 32
Leaf D8: Theta Range 3.1416 rad -> Bits: 32

--- Results ---
Header Overhead: 31.94 KB
Entropy Payload: 4.40 KB
Total Compressed: 36.34 KB
Original: 117.19 KB
Ratio: 3.22x


C:\Users\Denys\AppData\Local\Temp\ipykernel_22740\2824189438.py:190: RuntimeWarning: divide by zero encountered in scalar divide
  steps = val_range / error_budget


In [15]:
import numpy as np
from scipy.spatial.transform import Rotation as R
from scipy.optimize import least_squares
from collections import namedtuple

# --- Structures ---

class EllipsoidParams:
    def __init__(self, center, radii, rotation_matrix):
        self.center = center
        self.radii = radii
        self.rotation_matrix = rotation_matrix

class CompressionNode:
    def __init__(self, depth):
        self.depth = depth
        self.ellipsoid = None
        self.left = None
        self.right = None
        self.is_leaf = False

        # Leaf Data
        self.residuals_r = None
        self.residuals_theta = None
        self.residuals_phi = None

        # Metadata
        self.min_r, self.max_r = 0.0, 0.0
        self.min_theta, self.max_theta = 0.0, 0.0
        self.min_phi, self.max_phi = 0.0, 0.0

        # Bits
        self.bits_r, self.bits_theta, self.bits_phi = 0, 0, 0
        self.entropy_r, self.entropy_theta, self.entropy_phi = 0.0, 0.0, 0.0

Stats = namedtuple('Stats', ['header_bits', 'payload_bits'])

# --- Robust Geometry ---

def fit_ellipsoid_constrained(points):
    # Safety Check
    if len(points) < 4:
        return EllipsoidParams(np.mean(points, axis=0), np.ones(3), np.eye(3))

    # 1. PCA Init
    center = np.mean(points, axis=0)
    centered = points - center
    cov = np.cov(centered, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)

    # Sort descending
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    # Initial Radii: Clamp to avoid zero or infinity
    # We use log-radii for optimization, so we need safe initial values
    init_radii = 2.0 * np.sqrt(np.maximum(eigvals, 1e-4))
    init_radii = np.clip(init_radii, 0.01, 1000.0)

    # 2. Optimization
    def get_unit_error(params, p_target):
        c = params[0:3]
        r_log = params[3:6]
        rot_vec = params[6:9]

        rot_mat = R.from_rotvec(rot_vec).as_matrix()
        # Constraint: Prevent exponential overflow
        r_log = np.clip(r_log, -5, 5)
        radii_val = np.exp(r_log)

        # Transform
        p_c = p_target - c
        p_rot = p_c @ rot_mat
        p_scaled = p_rot / radii_val

        return np.linalg.norm(p_scaled, axis=1) - 1.0

    x0 = np.concatenate([center, np.log(init_radii), R.from_matrix(eigvecs).as_rotvec()])

    try:
        # Bounds for Center(unbounded), LogRadii(-5,5), Rot(unbounded)
        # Note: least_squares bounds is a tuple of arrays (min, max)
        # We leave center/rot unbounded (-np.inf, np.inf)
        lower_bounds = np.concatenate([[-np.inf]*3, [-5.0]*3, [-np.inf]*3])
        upper_bounds = np.concatenate([[np.inf]*3,  [5.0]*3,  [np.inf]*3])

        res = least_squares(get_unit_error, x0, bounds=(lower_bounds, upper_bounds), args=(points,), method='trf', max_nfev=20)

        opt_c = res.x[0:3]
        opt_radii = np.exp(np.clip(res.x[3:6], -5, 5))
        opt_rot = R.from_rotvec(res.x[6:9]).as_matrix()

        return EllipsoidParams(opt_c, opt_radii, opt_rot)
    except Exception as e:
        # Fallback to PCA if optimization fails
        return EllipsoidParams(center, init_radii, eigvecs)

def align_grid(points, ellipsoid):
    # Transform points to local
    p = points - ellipsoid.center
    p = p @ ellipsoid.rotation_matrix
    p = p / ellipsoid.radii

    # PCA on local points
    if len(p) < 3: return ellipsoid

    cov = np.cov(p, rowvar=False)
    _, eigvecs = np.linalg.eigh(cov)

    # Update rotation: Combined = Old * New
    new_rot = ellipsoid.rotation_matrix @ eigvecs
    return EllipsoidParams(ellipsoid.center, ellipsoid.radii, new_rot)

def transform_forward(points, ellipsoid):
    p = points - ellipsoid.center
    p = p @ ellipsoid.rotation_matrix
    p = p / ellipsoid.radii
    return p

# --- Tree ---

def build_tree(points, depth, max_depth):
    node = CompressionNode(depth)

    # Fit & Align
    node.ellipsoid = fit_ellipsoid_constrained(points)
    node.ellipsoid = align_grid(points, node.ellipsoid)

    next_points = transform_forward(points, node.ellipsoid)

    # Termination
    if depth >= max_depth or len(points) <= 4:
        node.is_leaf = True

        r = np.linalg.norm(next_points, axis=1)
        z_div_r = np.clip(next_points[:, 2] / (r + 1e-9), -1.0, 1.0)
        theta = np.arccos(z_div_r)
        phi = np.arctan2(next_points[:, 1], next_points[:, 0])

        node.residuals_r = r
        node.residuals_theta = theta
        node.residuals_phi = phi

        node.min_r, node.max_r = np.min(r), np.max(r)
        node.min_theta, node.max_theta = np.min(theta), np.max(theta)
        node.min_phi, node.max_phi = np.min(phi), np.max(phi)
        return node

    # Split (Spatial X-axis)
    # Since we aligned, X is principal axis
    sort_idx = np.argsort(next_points[:, 0])
    sorted_pts = next_points[sort_idx]
    mid = len(sorted_pts) // 2

    node.left = build_tree(sorted_pts[:mid], depth + 1, max_depth)
    node.right = build_tree(sorted_pts[mid:], depth + 1, max_depth)

    return node

# --- Analysis ---

def calculate_entropy(indices):
    if len(indices) == 0: return 0.0
    _, counts = np.unique(indices, return_counts=True)
    probs = counts / len(indices)
    return -np.sum(probs * np.log2(probs + 1e-9))

def analyze_tree(node, max_error_world, parent_scale=1.0):
    # Scale Propagation:
    # A local error of 1 unit becomes (1 * Radii) in parent space.
    # The max stretch this node introduces is max(Radii).
    # Total Stretch from Here -> World = ParentStretch * max(Radii)

    current_stretch = np.max(node.ellipsoid.radii)
    total_stretch = parent_scale * current_stretch

    # Header: 16 floats (10 ellipsoid + 6 ranges)
    header_bits = 16 * 32

    if node.is_leaf:
        # We need: local_error * total_stretch < max_error_world
        target_local = max_error_world / (total_stretch + 1e-9)

        # Helper
        def get_bits(rnge, target):
            if rnge < 1e-9: return 0
            steps = rnge / target
            if steps <= 1: return 1
            return int(np.clip(np.ceil(np.log2(steps)), 1, 32))

        # Bits Calculation
        node.bits_r = get_bits(node.max_r - node.min_r, target_local)
        node.bits_theta = get_bits(node.max_theta - node.min_theta, target_local)
        node.bits_phi = get_bits(node.max_phi - node.min_phi, target_local)

        # Quantization Simulation
        def quant(vals, vmin, vmax, bits):
            if bits == 0: return np.zeros(len(vals), dtype=int)
            norm = (vals - vmin) / (vmax - vmin)
            return np.round(np.clip(norm, 0, 1) * (2**bits - 1)).astype(int)

        q_r = quant(node.residuals_r, node.min_r, node.max_r, node.bits_r)
        q_t = quant(node.residuals_theta, node.min_theta, node.max_theta, node.bits_theta)
        q_p = quant(node.residuals_phi, node.min_phi, node.max_phi, node.bits_phi)

        count = len(node.residuals_r)

        # Entropy
        ent_bits = count * (calculate_entropy(q_r) + calculate_entropy(q_t) + calculate_entropy(q_p))

        return Stats(header_bits, ent_bits)

    else:
        l = analyze_tree(node.left, max_error_world, total_stretch)
        r = analyze_tree(node.right, max_error_world, total_stretch)
        return Stats(header_bits + l.header_bits + r.header_bits, l.payload_bits + r.payload_bits)

# --- Main ---
if __name__ == "__main__":
    np.random.seed(42)
    print("Generating Patch Data...")

    # Generate 5000 points on a surface patch (simulate 3D scan data)
    n = 5000
    u = np.random.uniform(0.4, 0.6, n)
    v = np.random.uniform(0.4, 0.6, n)
    theta = u * np.pi
    phi = v * 2 * np.pi

    x = np.sin(theta) * np.cos(phi) * 10.0
    y = np.sin(theta) * np.sin(phi) * 5.0
    z = np.cos(theta) * 2.0

    points = np.stack([x, y, z], axis=1)
    points += np.random.normal(0, 0.005, points.shape) # Noise

    # Calc Target Error
    diag = np.linalg.norm(np.max(points, axis=0) - np.min(points, axis=0))
    MAX_ERR = diag * 0.001 # 0.1% error

    print(f"Target Error: {MAX_ERR:.6f}")

    print("\n1. Building Tree (Depth 6)...")
    root = build_tree(points, 0, 6)

    print("2. Analyzing...")
    stats = analyze_tree(root, MAX_ERR)

    # Print Leaf Stats
    def print_leaves(node, count=[0]):
        if node.is_leaf and count[0] < 5:
            rng = node.max_theta - node.min_theta
            print(f"Leaf: Th_Rng={rng:.4f}, Bits R/T/P: {node.bits_r}/{node.bits_theta}/{node.bits_phi}")
            count[0] += 1
        if node.left: print_leaves(node.left, count)
        if node.right: print_leaves(node.right, count)

    print("\n--- Leaf Samples ---")
    print_leaves(root)

    # Totals
    total_bytes = (stats.header_bits + stats.payload_bits) / 8
    orig_bytes = n * 24

    print("\n--- Final Results ---")
    print(f"Header: {stats.header_bits/8/1024:.2f} KB")
    print(f"Payload: {stats.payload_bits/8/1024:.2f} KB")
    print(f"Total: {total_bytes/1024:.2f} KB")
    print(f"Ratio: {orig_bytes/total_bytes:.2f}x")

Generating Patch Data...
Target Error: 0.006431

1. Building Tree (Depth 6)...
2. Analyzing...

--- Leaf Samples ---
Leaf: Th_Rng=3.1257, Bits R/T/P: 32/31/31
Leaf: Th_Rng=3.1388, Bits R/T/P: 32/32/32
Leaf: Th_Rng=3.1401, Bits R/T/P: 32/32/32
Leaf: Th_Rng=3.0966, Bits R/T/P: 32/32/32
Leaf: Th_Rng=3.1416, Bits R/T/P: 32/32/32

--- Final Results ---
Header: 7.94 KB
Payload: 9.66 KB
Total: 17.60 KB
Ratio: 6.66x


In [16]:
import numpy as np
from collections import namedtuple

# --- Structures ---

class EllipsoidParams:
    def __init__(self, center, radii, rotation_matrix):
        self.center = center
        self.radii = radii
        self.rotation_matrix = rotation_matrix

class CompressionNode:
    def __init__(self, depth):
        self.depth = depth
        self.ellipsoid = None
        self.left = None
        self.right = None
        self.is_leaf = False

        # Leaf Data
        self.residuals_r = None
        self.residuals_theta = None
        self.residuals_phi = None

        # Ranges & Bits
        self.min_r, self.max_r = 0.0, 0.0
        self.min_theta, self.max_theta = 0.0, 0.0
        self.min_phi, self.max_phi = 0.0, 0.0
        self.bits_r, self.bits_theta, self.bits_phi = 0, 0, 0
        self.entropy_r, self.entropy_theta, self.entropy_phi = 0.0, 0.0, 0.0

Stats = namedtuple('Stats', ['header_bits', 'payload_bits'])

# --- Robust PCA Fitting ---

def fit_pca_ellipsoid(points):
    """
    Fits a tight Oriented Bounding Ellipsoid using PCA.
    This avoids the 'Infinite Radius' problem of surface fitting.
    """
    if len(points) < 4:
        return EllipsoidParams(np.mean(points, axis=0), np.ones(3)*0.001, np.eye(3))

    # 1. Center & Rotation (PCA)
    center = np.mean(points, axis=0)
    centered = points - center
    cov = np.cov(centered, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)

    # Sort descending (X=Largest Var, Z=Smallest Var)
    idx = np.argsort(eigvals)[::-1]
    rotation = eigvecs[:, idx]

    # 2. Project to determine tight Radii
    # Transform world -> unscaled local
    p_centered = points - center
    p_rotated = p_centered @ rotation

    # 3. Radii = Max Extent
    # We add a tiny epsilon to prevent division by zero in flat dimensions
    radii = np.max(np.abs(p_rotated), axis=0)
    radii = np.maximum(radii, 1e-6)

    return EllipsoidParams(center, radii, rotation)

def transform_forward(points, ellipsoid):
    p = points - ellipsoid.center
    p = p @ ellipsoid.rotation_matrix
    p = p / ellipsoid.radii
    return p

def transform_inverse(local_points, ellipsoid):
    p = local_points * ellipsoid.radii
    p = p @ ellipsoid.rotation_matrix.T
    p = p + ellipsoid.center
    return p

# --- Tree Building ---

def build_tree(points, depth, max_depth):
    node = CompressionNode(depth)

    # Fit Tight Ellipsoid (OBB)
    node.ellipsoid = fit_pca_ellipsoid(points)

    # Transform to Unit Cube [-1, 1]
    next_points = transform_forward(points, node.ellipsoid)

    # Termination
    if depth >= max_depth or len(points) <= 4:
        node.is_leaf = True

        # Convert Unit Cube points to Spherical for encoding
        # Note: Since PCA centers the data, r will range [0, ~1.73] (sqrt(3))
        r = np.linalg.norm(next_points, axis=1)
        z_div_r = np.clip(next_points[:, 2] / (r + 1e-9), -1.0, 1.0)
        theta = np.arccos(z_div_r)
        phi = np.arctan2(next_points[:, 1], next_points[:, 0])

        node.residuals_r = r
        node.residuals_theta = theta
        node.residuals_phi = phi

        node.min_r, node.max_r = np.min(r), np.max(r)
        node.min_theta, node.max_theta = np.min(theta), np.max(theta)
        node.min_phi, node.max_phi = np.min(phi), np.max(phi)
        return node

    # Split (Spatial X-axis of Local Frame)
    # PCA guarantees X is the axis of highest variance, so this is optimal.
    sort_idx = np.argsort(next_points[:, 0])
    sorted_pts = next_points[sort_idx]
    mid = len(sorted_pts) // 2

    node.left = build_tree(sorted_pts[:mid], depth + 1, max_depth)
    node.right = build_tree(sorted_pts[mid:], depth + 1, max_depth)
    return node

# --- Analysis & Decompression ---

def calculate_entropy(indices):
    if len(indices) == 0: return 0.0
    _, counts = np.unique(indices, return_counts=True)
    probs = counts / len(indices)
    return -np.sum(probs * np.log2(probs + 1e-9))

def analyze_tree(node, max_error_world, parent_scale=1.0):
    # Scale Prop: Max stretch is the largest radius
    current_stretch = np.max(node.ellipsoid.radii)
    total_stretch = parent_scale * current_stretch

    # Header: 16 floats (Ellipsoid + Ranges)
    header_bits = 16 * 32

    if node.is_leaf:
        # local_error * total_stretch < global_error
        target_local = max_error_world / (total_stretch + 1e-9)

        def get_bits(rnge, target):
            if rnge < 1e-9: return 0
            steps = rnge / target
            if steps <= 1: return 1
            return int(np.clip(np.ceil(np.log2(steps)), 1, 32))

        # Bits
        node.bits_r = get_bits(node.max_r - node.min_r, target_local)

        # For Angles: Arc Length error ~ r * angle_error.
        # Since r is normalized (max ~1.7), we can use r=1 approx or max_r.
        # safe_target_angle = target_local / max_r
        safe_r = max(node.max_r, 0.1)
        target_angle = target_local / safe_r

        node.bits_theta = get_bits(node.max_theta - node.min_theta, target_angle)
        node.bits_phi = get_bits(node.max_phi - node.min_phi, target_angle)

        # Quantize & Entropy
        def quant(vals, vmin, vmax, bits):
            if bits == 0: return np.zeros(len(vals), dtype=int)
            norm = (vals - vmin) / (vmax - vmin + 1e-9)
            return np.round(np.clip(norm, 0, 1) * (2**bits - 1)).astype(int)

        q_r = quant(node.residuals_r, node.min_r, node.max_r, node.bits_r)
        q_t = quant(node.residuals_theta, node.min_theta, node.max_theta, node.bits_theta)
        q_p = quant(node.residuals_phi, node.min_phi, node.max_phi, node.bits_phi)

        count = len(node.residuals_r)
        ent_bits = count * (calculate_entropy(q_r) + calculate_entropy(q_t) + calculate_entropy(q_p))

        return Stats(header_bits, ent_bits)

    else:
        l = analyze_tree(node.left, max_error_world, total_stretch)
        r = analyze_tree(node.right, max_error_world, total_stretch)
        return Stats(header_bits + l.header_bits + r.header_bits, l.payload_bits + r.payload_bits)

def decompress_recursive(node):
    if node.is_leaf:
        # Simulate Reconstruction
        # (Using midpoint of quantized buckets for simplicity in this check)
        local_x = node.residuals_r * np.sin(node.residuals_theta) * np.cos(node.residuals_phi)
        local_y = node.residuals_r * np.sin(node.residuals_theta) * np.sin(node.residuals_phi)
        local_z = node.residuals_r * np.cos(node.residuals_theta)
        local_points = np.stack([local_x, local_y, local_z], axis=1)

        return transform_inverse(local_points, node.ellipsoid)
    else:
        pts_left = decompress_recursive(node.left) if node.left else np.empty((0,3))
        pts_right = decompress_recursive(node.right) if node.right else np.empty((0,3))
        combined = np.concatenate([pts_left, pts_right], axis=0)
        return transform_inverse(combined, node.ellipsoid)

# --- Main ---
if __name__ == "__main__":
    np.random.seed(42)
    print("Generating Patch Data...")

    num_points = 5000
    u = np.random.uniform(0.45, 0.55, num_points) # 0.1 range
    v = np.random.uniform(0.45, 0.55, num_points) # 0.1 range
    theta = u * np.pi
    phi = v * 2 * np.pi

    # A small patch on a larger sphere
    x = np.sin(theta) * np.cos(phi) * 100.0
    y = np.sin(theta) * np.sin(phi) * 100.0
    z = np.cos(theta) * 100.0

    points = np.stack([x, y, z], axis=1)
    points += np.random.normal(0, 0.01, points.shape) # Noise

    # Verify dimensions
    diag = np.linalg.norm(np.max(points, axis=0) - np.min(points, axis=0))
    MAX_ERR = diag * 0.001
    print(f"Patch Size (Diag): {diag:.4f}")
    print(f"Target Error: {MAX_ERR:.6f}")

    print("\n1. Building Tree (Depth 8)...")
    root = build_tree(points, 0, 8)

    print("2. Analyzing...")
    stats = analyze_tree(root, MAX_ERR)

    def print_leaves(node, count=[0]):
        if node.is_leaf and count[0] < 5:
            print(f"Leaf D{node.depth}: Bits R/T/P: {node.bits_r}/{node.bits_theta}/{node.bits_phi} | Rng Phi: {node.max_phi-node.min_phi:.4f}")
            count[0] += 1
        if node.left: print_leaves(node.left, count)
        if node.right: print_leaves(node.right, count)

    print("\n--- Leaf Samples ---")
    print_leaves(root)

    total_bytes = (stats.header_bits + stats.payload_bits) / 8
    orig_bytes = num_points * 24

    print("\n--- Final Results ---")
    print(f"Header: {stats.header_bits/8/1024:.2f} KB")
    print(f"Payload: {stats.payload_bits/8/1024:.2f} KB")
    print(f"Total: {total_bytes/1024:.2f} KB")
    print(f"Ratio: {orig_bytes/total_bytes:.2f}x")

    # Verify Reconstruction
    print("\nVerifying Reconstruction...")
    rec = decompress_recursive(root)
    err = np.max(np.linalg.norm(points - rec, axis=1))
    print(f"Max Error: {err:.6f} (Target: {MAX_ERR:.6f})")

Generating Patch Data...
Patch Size (Diag): 69.4921
Target Error: 0.069492

1. Building Tree (Depth 8)...
2. Analyzing...

--- Leaf Samples ---
Leaf D8: Bits R/T/P: 11/11/14 | Rng Phi: 6.1386
Leaf D8: Bits R/T/P: 10/12/13 | Rng Phi: 5.9323
Leaf D8: Bits R/T/P: 10/12/13 | Rng Phi: 5.5094
Leaf D8: Bits R/T/P: 10/12/13 | Rng Phi: 5.4672
Leaf D8: Bits R/T/P: 10/11/13 | Rng Phi: 5.6081

--- Final Results ---
Header: 31.94 KB
Payload: 7.83 KB
Total: 39.77 KB
Ratio: 2.95x

Verifying Reconstruction...
Max Error: 62.301486 (Target: 0.069492)
